<a href="https://colab.research.google.com/github/ingkapat/Thai-Scam-Call-Detector/blob/main/clean2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
# === Cell 1: Setup ===
!pip install -q openai

from google.colab import drive, userdata, files
drive.mount('/content/drive')

import os, json, time, re, zipfile, random, threading
import concurrent.futures as cf
from collections import Counter
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

DRIVE_BASE = '/content/drive/MyDrive/ThaiScamCall'
OUT_DIR    = f'{DRIVE_BASE}/clean'
ZIP_AUDIO  = f'{DRIVE_BASE}/mp3_15s.zip'
EXT_AUDIO  = '/content/data_15s'
os.makedirs(OUT_DIR, exist_ok=True)

# OpenRouter client
gpt = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=userdata.get('OPENROUTER_KEY')
)
GPT_MODEL = 'openai/gpt-5-mini'   # 🔧 ลอง: openai/gpt-5-mini, anthropic/claude-sonnet-4.6

# ===== Config =====
MODE          = 'all'   # 🔧 'sample'(300,~$0.05) | 'all'(21K,~$2-3)
SAMPLE_N      = 300
GPT_WORKERS   = 80
PROMPT_VER    = 'v6'
REALISTIC_MIN = 3

print(f'✅ Setup OK | MODE={MODE} | model={GPT_MODEL} | prompt={PROMPT_VER}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup OK | MODE=all | model=openai/gpt-5-mini | prompt=v6


In [4]:
# === Cell 2: Upload dataset_cleaned.jsonl + dataset.jsonl (สำหรับ subtype map) ===
def full_text(d):
    return ' '.join(t['text'] for t in d['turns'])

# --- Step 1: Upload dataset_cleaned.jsonl (ลบ duplicate แล้ว ~21K) ---
print('📤 (1/2) Upload dataset_cleaned.jsonl (ลบ duplicate แล้ว)')
up1 = files.upload()
fname1 = list(up1.keys())[0]

data = []
for line in up1[fname1].decode('utf-8').splitlines():
    if line.strip(): data.append(json.loads(line))

for i, d in enumerate(data):
    d['line']    = i + 1
    d['conv_id'] = f'conv_{i+1:05d}_label{d["label"]}'

print(f'\n✅ Cleaned: {len(data)} บท | Keys: {list(data[0].keys())}')

# --- Step 2: Upload dataset.jsonl (มี subtype — สำหรับ map เข้า data) ---
has_subtype = 'subtype' in data[0]
if has_subtype:
    print('\n✅ dataset_cleaned มี subtype field อยู่แล้ว — ข้ามขั้นตอนนี้')
else:
    print('\n📤 (2/2) Upload dataset.jsonl (มี subtype — สำหรับ map)')
    up2 = files.upload()
    fname2 = list(up2.keys())[0]

    orig = []
    for ln in up2[fname2].decode('utf-8').splitlines():
        if ln.strip(): orig.append(json.loads(ln))
    print(f'   Original: {len(orig)} บท')

    # build map: full_text → subtype
    orig_map = {}
    for d in orig:
        key = full_text(d)
        if key not in orig_map:
            orig_map[key] = d.get('subtype', 'NONE')

    matched = 0
    for d in data:
        sub = orig_map.get(full_text(d), 'NONE')
        d['subtype'] = sub
        if sub != 'NONE': matched += 1
    print(f'   ✅ matched subtype: {matched}/{len(data)} ({matched/len(data)*100:.1f}%)')

# --- สรุป ---
subs = Counter(d.get('subtype', 'NONE') for d in data)
print(f'\nSubtype distribution (top 10):')
for s, c in subs.most_common(10): print(f'  {s}: {c}')

print(f'\nLabel:')
print(pd.Series([d['label'] for d in data]).value_counts()
      .rename({0:'not_scam', 1:'scam'}))

📤 (1/2) Upload dataset_cleaned.jsonl (ลบ duplicate แล้ว)


Saving dataset_cleaned.jsonl to dataset_cleaned.jsonl

✅ Cleaned: 21352 บท | Keys: ['label', 'turns', 'line', 'conv_id']

📤 (2/2) Upload dataset.jsonl (มี subtype — สำหรับ map)


Saving dataset.jsonl to dataset.jsonl
   Original: 22156 บท
   ✅ matched subtype: 5393/21352 (25.3%)

Subtype distribution (top 10):
  NONE: 15959
  otp_hijack: 487
  otp_thai: 462
  reset_code: 435
  sextortion: 333
  gov_notice: 267
  loan_scam: 264
  company_hr: 256
  bank_real: 204
  real_parcel: 170

Label:
not_scam    11212
scam        10140
Name: count, dtype: int64


In [27]:
# === Cell 3: GPT check function (prompt v6) ===
SYSTEM = """คุณตรวจสอบบทสนทนาโทรศัพท์ (ฝึก AI ตรวจ scam call ไทย) จาก transcript
label เดิม: 0=ปกติ(not_scam), 1=scam — เชื่อ label เดิมไว้ก่อน

ตอบเฉพาะ JSON object (ไม่มี text อื่น):
{"true_label": 0หรือ1, "realistic": 0-10, "action": "keep"/"relabel"/"remove", "reason": "สั้นๆ"}

⚠️ relabel asymmetric: ก้ำกึ่ง → keep (default)

🎯 SCAM ชัด (relabel 0→1):
- ตำรวจ/สรรพากร/ธนาคาร ขู่/อายัด → โอนเงิน
- AnyDesk/TeamViewer, Line ID @police-xxx
- ลงทุนผลตอบแทน 30-50%/เดือน

✅ HARD NEGATIVE — label=0 ⚠️ ห้าม relabel เป็น scam:
- real_otp: ธนาคารโทรกลับ confirm OTP + reference number (user เริ่ม transaction เอง)
- friend_new_number: เพื่อนจริงเปลี่ยนเบอร์ + ยืมเงิน + มี personal context
- friend_borrow/colleague_expense/family_help: ยืมเงินจริง มี context
- verification: ระบบส่ง OTP ให้ user
- official: ธนาคาร/หน่วยงานจริง ไม่ขอ OTP-ลิงก์
- delivery/legit_promo: ไม่ขอเงิน-OTP

action=remove = บทเสีย (breathing/instruction หลุด/สั้นเกิน)
realistic ต่ำ = ฉากไม่น่าเกิดจริง"""

def gpt_check(d):
    txt = '\n'.join(f"{t['speaker']}: {t['text']}" for t in d['turns'])
    for attempt in range(4):
        try:
            r = gpt.chat.completions.create(
                model=GPT_MODEL,
                messages=[{'role':'system','content':SYSTEM},
                          {'role':'user','content':f'label เดิม={d["label"]}\nบทสนทนา:\n{txt}\n\nตอบเฉพาะ JSON.'}],
                temperature=0,
                max_tokens=1500,   # 🔧 จาก 150 → 1500 (กัน reasoning model ตัด)
            )
            content = r.choices[0].message.content
            if not content:
                fr = r.choices[0].finish_reason
                return {'action':'error','reason':f'empty content (finish={fr})'}
            m = re.search(r'\{[^{}]*\}', content, re.DOTALL)
            if not m:
                return {'action':'error','reason':f'no json: {content[:50]}'}
            return json.loads(m.group())
        except Exception as e:
            if attempt == 3:
                return {'action':'error','reason':str(e)[:80]}
            time.sleep(2 ** attempt)

print('Test:', gpt_check(data[0]))
print('Model:', GPT_MODEL)

Test: {'true_label': 1, 'realistic': 10, 'action': 'keep', 'reason': 'อ้างเป็นธนาคารขอรหัส OTP โดยตรง เป็นลักษณะหลอกเอา OTP ชัดเจน'}
Model: openai/gpt-5-mini


In [28]:
# === Cell 4: Run GPT (parallel + checkpoint + resume errors) ===
target = (random.Random(42).sample(data, min(SAMPLE_N, len(data)))
          if MODE == 'sample' else data)

CKPT = f'{OUT_DIR}/gpt_ckpt_{MODE}_{PROMPT_VER}.jsonl'
done = {}
if os.path.exists(CKPT):
    for ln in open(CKPT, encoding='utf-8'):
        if ln.strip():
            o = json.loads(ln); done[o['conv_id']] = o

todo = [d for d in target
        if d['conv_id'] not in done or done[d['conv_id']].get('action') == 'error']
print(f'MODE={MODE} | target={len(target)} | done={len(done)} | todo={len(todo)}')

lock = threading.Lock()
ckf  = open(CKPT, 'a', encoding='utf-8')
def work(d):
    r = gpt_check(d)
    rec = {'conv_id':d['conv_id'],
           'true_label':r.get('true_label', d['label']),
           'realistic':r.get('realistic', 5),
           'action':r.get('action', 'keep'),
           'reason':r.get('reason', '')}
    with lock:
        ckf.write(json.dumps(rec, ensure_ascii=False) + '\n'); ckf.flush()
    return rec
with cf.ThreadPoolExecutor(GPT_WORKERS) as ex:
    list(tqdm(ex.map(work, todo), total=len(todo), desc='GPT'))
ckf.close()

# merge
done = {}
for ln in open(CKPT, encoding='utf-8'):
    if ln.strip():
        o = json.loads(ln); done[o['conv_id']] = o
for d in target:
    o = done.get(d['conv_id'], {})
    d['gpt_label'] = o.get('true_label', d['label'])
    d['realistic'] = o.get('realistic', 5)
    d['action']    = o.get('action', 'keep')
    d['reason']    = o.get('reason', '')

# สรุป
acts = Counter(d['action'] for d in target)
up   = sum(d['action']=='relabel' and d['label']==0 and d['gpt_label']==1 for d in target)
down = sum(d['action']=='relabel' and d['label']==1 and d['gpt_label']==0 for d in target)
low  = sum(d['realistic'] < REALISTIC_MIN for d in target)
print('\n=== Action ===')
for a,c in acts.items(): print(f'  {a}: {c} ({c/len(target)*100:.1f}%)')
print(f'  relabel 0→1: {up} | 1→0: {down}')
print(f'  realistic<{REALISTIC_MIN}: {low}')
print(f'  errors: {acts.get("error",0)}')


MODE=all | target=21352 | done=0 | todo=21352


GPT:   0%|          | 0/21352 [00:00<?, ?it/s]


=== Action ===
  keep: 19369 (90.7%)
  remove: 1742 (8.2%)
  relabel: 240 (1.1%)
  error: 1 (0.0%)
  relabel 0→1: 198 | 1→0: 42
  realistic<3: 726
  errors: 1


In [29]:
# === Cell 5: Inspect + save QA CSV ===
df = pd.DataFrame([{
    'conv_id':d['conv_id'], 'label':d['label'], 'gpt_label':d['gpt_label'],
    'subtype':d.get('subtype','NONE'),
    'realistic':d['realistic'],
    'action':d['action'], 'reason':d['reason'], 'text':full_text(d)[:90],
} for d in target])

print('=== RELABEL ===')
display(df[df.action=='relabel'][['conv_id','subtype','label','gpt_label','reason','text']].head(20))
print('=== REMOVE ===')
display(df[df.action=='remove'][['conv_id','reason','text']].head(10))

# QA CSV
rl = df[df.action=='relabel'].copy()
rl['full_text'] = [full_text(d) for d in target if d['action']=='relabel']
rl['correct(T/F)'] = ''
rl[['conv_id','subtype','label','gpt_label','reason','full_text','correct(T/F)']].to_csv(
    f'{OUT_DIR}/relabels_review_{MODE}.csv', index=False, encoding='utf-8-sig')
print(f'\n💾 saved relabels_review_{MODE}.csv ({len(rl)} ตัว)')


=== RELABEL ===


,conv_id,subtype,label,gpt_label,reason,text
34,conv_00035_label0,NONE,0,1,แอบอ้างธนาคารขอ OTP ทางโทรศัพท์ — เป็นพฤติกรรม...,พี่จ๋า สวัสดีครับ นี่ธนาคารนะครับ ค่ะ มีอะไรเห...
123,conv_00124_label0,NONE,0,1,อ้างเป็นธนาคารขอ OTP เพื่อยืนยันรายการ ซึ่งเป็...,ขอโทษครับ คุณ ชื่อ ดิฉันจากธนาคารนะครับ ต้องกา...
288,conv_00289_label0,NONE,0,1,คนโทรอ้างเป็นธนาคารขอให้ส่ง OTP — รูปแบบการหลอ...,พี่จ๋า สวัสดีครับ ผมจากธนาคารนะครับ สวัสดีครับ...
344,conv_00345_label0,NONE,0,1,โทรอ้างเป็นธนาคารขอ OTP เพื่อยืนยันธุรกรรม — ร...,เฮ้ย สวัสดีครับ ผมจากธนาคารนะครับ โทรมาเพื่อยื...
363,conv_00364_label1,NONE,1,0,ลักษณะเป็นเพื่อนเปลี่ยนเบอร์ขอยืมเงิน (friend_...,ฮัลโหลค่ะ แก ไอ้จูนนะ เปลี่ยนเบอร์ใหม่แล้วนะ อ...
404,conv_00405_label1,romance,1,0,การทักทายส่วนตัว ดูเป็นเพื่อน/คนในครอบครัว ไม่...,พี่จ๋า รักหนูนะ ทำงานอยู่ที่อเมริกาเลยเหรอ ค่ะ...
473,conv_00474_label0,real_otp,0,1,อ้างเป็นธนาคารขอ OTP เพื่อยืนยันธุรกรรม — รูปแ...,เฮ้ย สวัสดีครับ คุณสมปองใช่ไหมครับ ครับ ผมเอง ...
491,conv_00492_label0,NONE,0,1,อ้างเป็นธนาคารขอ OTP ซึ่งเป็นพฤติกรรมฉ้อโกงชัดเจน,เห้ สวัสดีครับ คุณนิดนะครับ จากธนาคารครับ ขอกว...
519,conv_00520_label0,real_otp,0,1,อ้างเป็นธนาคารขอ OTP ทางโทรศัพท์ — พฤติกรรมสคา...,หวัดดีครับ นี่ธนาคารนะครับ อ่าว สวัสดีครับ เรา...
533,conv_00534_label1,loan_scam,1,0,เป็นสายขายโปรโมชันสินเชื่อ ไม่มีการขอเงิน/OTP/...,ฮัลโหลค่ะ ยินดีต้อนรับสู่บริการสินเชื่อของเรา ...


=== REMOVE ===


,conv_id,reason,text
1,conv_00002_label0,บทสั้นเกิน ไม่มีบริบทเพียงพอ,เฮ้ย อ่า ก็ คือ ไม่รู้ว่า อืม แกอยู่ไหน
7,conv_00008_label0,สั้นเกิน ไม่มีบริบทหรือข้อมูลเพียงพอ,เฮ้ย แก คือ ไม่รู้ว่าทำไม
9,conv_00010_label0,บทสั้นเกิน ไม่มีเนื้อหาเพียงพอในการประเมิน,ฮัลโหล มีใครอยู่ไหม
10,conv_00011_label0,บทสั้นเกิน ไม่มีเนื้อหาหรือบริบทบ่งชี้การหลอกลวง,เฮ้ย มีใครอยู่ปลายสายป่าว
32,conv_00033_label0,บทสั้นเกิน ไม่มีเนื้อหาเกี่ยวกับการหลอกลวง,เฮ้ย แก ฟังไม่ค่อยชัดเลย อ่ะ มีเสียงสะท้อน อืม...
33,conv_00034_label0,บทสั้นเกิน ไม่พอข้อมูลสำหรับประเมิน,คุณครับ นี่โรงพยาบาลนะคะ ต้องการนัดหมายตรวจสุข...
37,conv_00038_label0,สั้นเกิน ไม่มีสัญญาณสแปมหรือบริบทเพียงพอ,เฮ้ย แก ได้ยินป่ะ เสียงมันแปลกๆ อ่ะ อือ ไม่ค่อ...
55,conv_00056_label0,ข้อความสั้นเกิน เป็นการแจ้งรหัสยืนยัน (OTP) ไม...,"เรียนผู้ใช้, รหัสยืนยันของคุณคือ 123456."
57,conv_00058_label1,สั้นเกินไป ไม่มีบริบทพอเพื่อยืนยันว่าเป็น scam...,ขอโทษครับ คุณกำลังมีปัญหากับบัญชีของคุณนะครับ
60,conv_00061_label0,สั้นเกิน ไม่มีข้อมูลพอให้ประเมิน,ฮัลโหล ใครครับ



💾 saved relabels_review_all.csv (240 ตัว)


In [30]:
# === Cell 6: วัด Hard Negative noise ratio (สำคัญ!) ===
PROTECTED_SUBTYPES = {
    'real_otp','real_job','friend_borrow','real_parcel','friend_new_number',
    'colleague_expense','family_help',
    'otp_thai','otp_english','reset_code',
    'bank_real','gov_notice','hospital','company_hr',
    'kerry_courier','food_delivery','ems_post','lazada_shopee',
    'ais_promo','true_promo','dtac_promo','bank_offer','insurance_offer',
}

hn_in_sample = [d for d in target if d.get('subtype') in PROTECTED_SUBTYPES]
hn_wrong = [d for d in hn_in_sample
            if d['action']=='relabel' and d['label']==0 and d['gpt_label']==1]
total_relabel = [d for d in target if d['action']=='relabel']
noise_ratio = len(hn_wrong)/max(len(total_relabel),1)*100

print(f'📊 Hard Negative Test ({PROMPT_VER} + {GPT_MODEL})')
print(f'   HN ใน sample:        {len(hn_in_sample)}')
print(f'   HN ถูก relabel ผิด:   {len(hn_wrong)}')
print(f'   Total relabel:       {len(total_relabel)}')
print(f'   ❗ Noise ratio:       {noise_ratio:.1f}%')

if noise_ratio < 20:    print('\n✅ ผ่าน! Run full ได้')
elif noise_ratio < 40:  print('\n⚠️ ลอง model: openai/gpt-5-mini')
else:                   print('\n❌ ลอง model: anthropic/claude-sonnet-4.6')

print('\n=== HN ที่ผิด ===')
for d in hn_wrong[:10]:
    print(f"[{d['conv_id']}] {d.get('subtype')}: {d['reason'][:60]}")

wrong_subs = Counter(d.get('subtype') for d in hn_wrong)
print('\n=== Subtype ผิดเยอะสุด ===')
for s, c in wrong_subs.most_common(5):
    print(f'  {s}: {c}')


📊 Hard Negative Test (v6 + openai/gpt-5-mini)
   HN ใน sample:        3073
   HN ถูก relabel ผิด:   56
   Total relabel:       240
   ❗ Noise ratio:       23.3%

⚠️ ลอง model: openai/gpt-5-mini

=== HN ที่ผิด ===
[conv_00474_label0] real_otp: อ้างเป็นธนาคารขอ OTP เพื่อยืนยันธุรกรรม — รูปแบบหลอกขโมยเงิน
[conv_00520_label0] real_otp: อ้างเป็นธนาคารขอ OTP ทางโทรศัพท์ — พฤติกรรมสคามชัดเจน (ธนาคา
[conv_00582_label0] real_otp: อ้างเป็นธนาคารขอรหัส OTP ทางโทรศัพท์ ซึ่งธนาคารไม่ขอ OTP ทาง
[conv_03167_label0] real_otp: แอบอ้างเป็นธนาคารขอให้ผู้ใช้ส่ง OTP ซึ่งเป็นพฤติกรรมฟิชชิ่ง/
[conv_03313_label0] real_otp: ขอ OTP ทางโทรศัพท์เพื่อยืนยันรายการ — พฤติกรรมคลาสสิกของ sca
[conv_03708_label0] real_otp: ผู้โทรอ้างเป็นธนาคารขอ OTP เพื่อยืนยันธุรกรรม — รูปแบบหลอกขอ
[conv_03731_label0] real_otp: ผู้โทรขอ OTP พร้อมหมายเลขอ้างอิง — ลักษณะการหลอกขอรหัสเพื่อถ
[conv_04538_label0] real_otp: ผู้โทรขอ OTP อ้างการทำรายการบัญชี — รูปแบบหลอกขอรหัสเพื่อโอน
[conv_04869_label0] real_otp: ผู้โทรอ้างเป็นธนาคารขอ OTP ทา

In [31]:
# === Cell 7: Apply → dataset_relabeled.jsonl + mp3_15s_clean.zip ===
APPLY = True   # 🔧 True เมื่อ MODE='all' + ตรวจผลแล้ว

if APPLY and MODE == 'all':
    clean = []; n_rm = 0; n_rl = 0; rm_lines = set()
    for d in data:
        if d['action'] == 'remove' or d['realistic'] < REALISTIC_MIN:
            n_rm += 1; rm_lines.add(d['line']); continue
        lbl = d['gpt_label'] if d['action'] == 'relabel' else d['label']
        if d['action'] == 'relabel': n_rl += 1
        clean.append({'label': lbl, 'turns': d['turns']})

    local_jsonl = 'dataset_relabeled.jsonl'
    with open(local_jsonl, 'w', encoding='utf-8') as f:
        for d in clean: f.write(json.dumps(d, ensure_ascii=False) + '\n')
    !cp {local_jsonl} {OUT_DIR}/{local_jsonl}

    if os.path.exists(ZIP_AUDIO):
        if not os.path.exists(EXT_AUDIO) or len(os.listdir(EXT_AUDIO)) < 1000:
            print('Extracting audio...')
            with zipfile.ZipFile(ZIP_AUDIO) as z: z.extractall(EXT_AUDIO)
        keep_au = [f for f in os.listdir(EXT_AUDIO) if f.endswith('.mp3')
                   and int(re.search(r'conv_(\d+)_', f).group(1)) not in rm_lines]
        local_zip = 'mp3_15s_clean.zip'
        with zipfile.ZipFile(local_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
            for fn in tqdm(keep_au, desc='zip'): zf.write(f'{EXT_AUDIO}/{fn}', arcname=fn)
        !cp {local_zip} {DRIVE_BASE}/{local_zip}
        print(f'✅ mp3_15s_clean.zip: {len(keep_au)} ไฟล์')

    print(f'✅ dataset_relabeled.jsonl: {len(clean)} บท (remove {n_rm}, relabel {n_rl})')
else:
    print('⏸️ ตั้ง MODE=all + APPLY=True')


Extracting audio...


zip:   0%|          | 0/19591 [00:00<?, ?it/s]

✅ mp3_15s_clean.zip: 19591 ไฟล์
✅ dataset_relabeled.jsonl: 19603 บท (remove 1749, relabel 239)


In [32]:
# === Cell 8: Report ===
hn_wrong_n = sum(1 for d in target
    if d.get('subtype') in PROTECTED_SUBTYPES
    and d['action']=='relabel' and d['label']==0 and d['gpt_label']==1)
total_rl = sum(1 for d in target if d['action']=='relabel')

report = {
    'mode': MODE,
    'model': GPT_MODEL,
    'prompt_version': PROMPT_VER,
    'n_total': len(target),
    'keep': sum(d['action']=='keep' for d in target),
    'remove_broken': sum(d['action']=='remove' for d in target),
    'remove_unrealistic': sum(d['action']!='remove' and d['realistic']<REALISTIC_MIN for d in target),
    'relabel': total_rl,
    'relabel_0to1': sum(d['action']=='relabel' and d['label']==0 and d['gpt_label']==1 for d in target),
    'relabel_1to0': sum(d['action']=='relabel' and d['label']==1 and d['gpt_label']==0 for d in target),
    'hard_negative_wrong': hn_wrong_n,
    'hard_negative_noise_pct': round(hn_wrong_n/max(total_rl,1)*100, 2),
}
with open(f'{OUT_DIR}/clean_report_{MODE}.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print(json.dumps(report, indent=2, ensure_ascii=False))


{
  "mode": "all",
  "model": "openai/gpt-5-mini",
  "prompt_version": "v6",
  "n_total": 21352,
  "keep": 19369,
  "remove_broken": 1742,
  "remove_unrealistic": 7,
  "relabel": 240,
  "relabel_0to1": 198,
  "relabel_1to0": 42,
  "hard_negative_wrong": 56,
  "hard_negative_noise_pct": 23.33
}
